In [1]:
!pip install gymnasium

In [2]:
import torch

# Configurem el dispositiu per fer servir CUDA si està disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Estàs fent servir: {device}")
if torch.cuda.is_available():
    print(f"Model de GPU: {torch.cuda.get_device_name(0)}")

Estàs fent servir: cuda
Model de GPU: Tesla T4


In [5]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
from env import MountainCarCustomized

# --- CONFIGURACIÓ DE DISPOSITIU (CUDA) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f">>> Fent servir dispositiu: {device}")

# --- PARÀMETRES DE L'AGENT ---
GAMMA = 0.98
LEARNING_RATE = 1e-3
MEMORY_CAPACITY = 500_000
BATCH_SIZE = 128
K_HER = 4  # Metes virtuals per cada pas real

class NeuralNet(nn.Module):
    """ Xarxa neuronal que rep estat + meta i prediu Q-values """
    def __init__(self, input_size, action_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, action_size)
        )

    def forward(self, state, goal):
        x = torch.cat([state, goal], dim=-1)
        return self.network(x)

class HER_Manager:
    def __init__(self, s_dim, g_dim, a_dim):
        self.actions = a_dim
        self.memory = deque(maxlen=MEMORY_CAPACITY)

        # Enviar models a la GPU
        self.q_net = NeuralNet(s_dim + g_dim, a_dim).to(device)
        self.target_net = NeuralNet(s_dim + g_dim, a_dim).to(device)
        self.target_net.load_state_dict(self.q_net.state_dict())

        self.opt = optim.Adam(self.q_net.parameters(), lr=LEARNING_RATE)
        self.epsilon = 1.0

    def select_action(self, s, g):
        if random.random() < self.epsilon:
            return random.randint(0, self.actions - 1)

        with torch.no_grad():
            # Convertir a tensor i enviar a GPU
            s_t = torch.as_tensor(s, dtype=torch.float32, device=device).unsqueeze(0)
            g_t = torch.as_tensor(g, dtype=torch.float32, device=device).unsqueeze(0)
            return self.q_net(s_t, g_t).argmax().item()

    def update_model(self):
        if len(self.memory) < BATCH_SIZE:
            return

        batch = random.sample(self.memory, BATCH_SIZE)
        s, a, r, sn, d, g = zip(*batch)

        # Creació de tensors directament al dispositiu (més ràpid)
        s = torch.tensor(np.array(s), dtype=torch.float32, device=device)
        a = torch.tensor(a, dtype=torch.long, device=device).unsqueeze(1)
        r = torch.tensor(r, dtype=torch.float32, device=device)
        sn = torch.tensor(np.array(sn), dtype=torch.float32, device=device)
        d = torch.tensor(d, dtype=torch.float32, device=device)
        g = torch.tensor(np.array(g), dtype=torch.float32, device=device)

        # Q actual
        q_val = self.q_net(s, g).gather(1, a).squeeze()

        # Q següent amb Target Net
        with torch.no_grad():
            max_next_q = self.target_net(sn, g).max(1)[0]
            expected_q = r + GAMMA * max_next_q * (1 - d)

        loss = nn.MSELoss()(q_val, expected_q)

        self.opt.zero_grad()
        loss.backward()
        self.opt.step()

        # Update suau del target (Soft Update)
        for t, p in zip(self.target_net.parameters(), self.q_net.parameters()):
            t.data.copy_(0.95 * t.data + 0.05 * p.data)

def run_experiment(limit_dist, total_episodes=2000):
    print(f"\n--- EXPERIMENT: DIST_LIMIT = {limit_dist} ---")
    env = MountainCarCustomized()

    s_dim = env.observation_space.shape[0]
    g_dim = 1
    a_dim = env.action_space.n

    agent = HER_Manager(s_dim, g_dim, a_dim)
    real_goal_pos = np.array([env.env.unwrapped.goal_position], dtype=np.float32)

    wins = 0

    for ep in range(1, total_episodes + 1):
        obs, _ = env.reset()
        history = []
        total_reward = 0
        done = False

        # Fase 1: Recollida de dades
        while not done:
            action = agent.select_action(obs, real_goal_pos)
            n_obs, rew, term, trunc, _ = env.step(action)

            history.append((obs, action, rew, n_obs))
            total_reward += rew
            obs = n_obs
            done = term or trunc

        if history[-1][2] > 0: wins += 1

        # Fase 2: Replay Buffer (Normal + HER)
        for i, (s, a, r, sn) in enumerate(history):
            # Transició real
            success = abs(sn[0] - real_goal_pos[0]) < limit_dist
            reward = 200.0 if success else 0.0
            agent.memory.append((s, a, reward, sn, float(success), real_goal_pos))

            # Transicions HER (Hindsight)
            future_steps = history[i+1:]
            if future_steps:
                samples = random.sample(future_steps, min(len(future_steps), K_HER))
                for _, _, _, s_future in samples:
                    v_goal = np.array([s_future[0]], dtype=np.float32)
                    v_success = abs(sn[0] - v_goal[0]) < limit_dist
                    v_reward = 200.0 if v_success else 0.0
                    agent.memory.append((s, a, v_reward, sn, float(v_success), v_goal))

        # Fase 3: Aprenentatge
        for _ in range(len(history)):
            agent.update_model()

        agent.epsilon = max(agent.epsilon * 0.998, 0.05)

        if ep % 100 == 0:
            success_rate = (wins / 100) * 100
            print(f"Ep {ep} | Reward: {total_reward:.1f} | Èxit: {success_rate:.1f}% | Eps: {agent.epsilon:.2f}")
            wins = 0

if __name__ == "__main__":
    # Pots triar quin dels dos executar o fer-los seguits
    run_experiment(limit_dist=0.15, total_episodes=2000)
    # run_experiment(limit_dist=0.15, total_episodes=2000)

>>> Fent servir dispositiu: cuda

--- EXPERIMENT: DIST_LIMIT = 0.15 ---
Ep 100 | Reward: 0.0 | Èxit: 0.0% | Eps: 0.82
Ep 200 | Reward: 0.0 | Èxit: 0.0% | Eps: 0.67
Ep 300 | Reward: 0.0 | Èxit: 0.0% | Eps: 0.55
Ep 400 | Reward: 0.0 | Èxit: 0.0% | Eps: 0.45
Ep 500 | Reward: 0.0 | Èxit: 0.0% | Eps: 0.37
Ep 600 | Reward: 0.0 | Èxit: 0.0% | Eps: 0.30
Ep 700 | Reward: 200.0 | Èxit: 5.0% | Eps: 0.25
Ep 800 | Reward: 200.0 | Èxit: 82.0% | Eps: 0.20
Ep 900 | Reward: 200.0 | Èxit: 92.0% | Eps: 0.17
Ep 1000 | Reward: 200.0 | Èxit: 95.0% | Eps: 0.14
Ep 1100 | Reward: 200.0 | Èxit: 83.0% | Eps: 0.11
Ep 1200 | Reward: 200.0 | Èxit: 95.0% | Eps: 0.09
Ep 1300 | Reward: 200.0 | Èxit: 89.0% | Eps: 0.07
Ep 1400 | Reward: 200.0 | Èxit: 96.0% | Eps: 0.06
Ep 1500 | Reward: 200.0 | Èxit: 92.0% | Eps: 0.05
Ep 1600 | Reward: 200.0 | Èxit: 92.0% | Eps: 0.05
Ep 1700 | Reward: 200.0 | Èxit: 98.0% | Eps: 0.05
Ep 1800 | Reward: 200.0 | Èxit: 90.0% | Eps: 0.05
Ep 1900 | Reward: 200.0 | Èxit: 92.0% | Eps: 0.05
Ep 200